# Reproducing Hulsey et al. (2024) from dandiset 000678

**Paper.** Hulsey D, Zumwalt K, Mazzucato L, McCormick DA, Jaramillo S (2024). Decision-making dynamics are predicted by arousal and uninstructed movements. *Cell Reports* 43(1), 113709. [doi:10.1016/j.celrep.2024.113709](https://doi.org/10.1016/j.celrep.2024.113709). The claims file was written from the accepted author manuscript in PubMed Central (PMC11016285) with its supplement; this attempt read the same text and the paper's analysis code (Zenodo [10.5281/zenodo.10306018](https://doi.org/10.5281/zenodo.10306018), and github.com/sjara/uobrainflex at commit ef48985192e4af332c64f9d2e70ddcfc78659040).

**Dandiset.** [DANDI:000678](https://dandiarchive.org/dandiset/000678/0.231004.2146), version 0.231004.2146, streamed from the archive. This is the version the paper cites (Data and code availability, and the Key resources table). A later version, 0.261020.0029, has the same 391 assets with the same asset identifiers and sizes; it differs only in the archive metadata (keywords, affiliations, ORCIDs), so the choice does not change the data.

**Claims.** `claims/000678/10.1016-j.celrep.2024.113709.yaml` (29 claims). The GLM-HMM and the analyses after it follow the paper's code, reimplemented in `hulsey.py`; where the claims file leaves a detail open, the passage used is cited in the section or in the helper's docstring.

| Claim | Tested here |
|---|---|
| cohort-inclusion | tested |
| training-duration | blocked: data missing (no sessions before stage 5) |
| distractor-response-rate | tested |
| figs2-n-states | tested |
| fig1g-example-mouse-prediction | tested |
| fig2a-state-types | tested |
| fig2b-optimal-occupancy | tested |
| fig2c-dwell-times | tested |
| fig2d-prediction-accuracy | tested |
| fig3c-inverted-u-pupil | tested |
| fig3d-optimal-pupil-modality | tested |
| fig3e-u-disengaged-pupil | tested |
| figs4-r2-individual-movement | tested |
| fig3f-movement-index-inverted-u | tested |
| fig3f-movement-index-u-disengaged | tested |
| figs5-movement-index-correlations | tested |
| fig3-distance-from-optimal-pupil | tested |
| fig3-raw-face-energy-trend | tested |
| fig3-raw-locomotion-no-difference | tested |
| fig4a-lower-variability | tested |
| fig4b-variability-at-transitions | tested |
| fig4d-pupil-movement-decoupling | tested |
| figs6-individual-measure-decoupling | tested |
| fig5c-state-classification | tested (fewer shuffle runs) |
| figs7-pupil-best-raw-measure | tested (fewer shuffle runs) |
| fig5c-movement-index-best | tested (fewer shuffle runs) |
| figs8a-regression-weights | tested |
| figs8b-pupil-movement-interaction | tested |
| figs1-iti-independence | tested |

The notebook takes about 27 minutes on 4 cores: about 3 to stream the 391 files, 15 for the cross-validated GLM-HMM fits and 7 for the state decoders. Random seeds are fixed, and each claim's section has its own generator.

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_v] = "1"  # many small fits run in parallel processes; one linear-algebra thread each
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import hulsey

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = hulsey.list_assets()
print(f"dandiset {hulsey.DANDISET} version {hulsey.VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e9:.2f} GB")
trials_all, alltrials, sessions = hulsey.load_all(assets)
print(f"streamed {len(sessions)} files in {(time.time() - started) / 60:.1f} min")

## Data check

Each file holds one session: a trials table, the measures of the face camera (online and post hoc pupil, face and whisker motion energy, sampled at about 29.5 Hz), wheel speed (about 100 Hz), and lick and reward times. The checks of the metadata against the values are stated under each output.

In [ ]:
meta = hulsey.version_metadata()
print("Archive metadata: name:", meta["name"])
print("  numberOfSubjects:", meta.get("assetsSummary", {}).get("numberOfSubjects"),
      "| numberOfFiles:", meta.get("assetsSummary", {}).get("numberOfFiles"))
print("\nSessions by subject, task (session_type), training stage:")
by = sessions.assign(task=sessions.session_type).groupby(["subject", "task"]).agg(
    sessions=("path", "size"), S5=("stage", lambda x: int((x == "S5").sum())),
    S6=("stage", lambda x: int((x == "S6").sum())),
    no_post_hoc_pupil=("pupil_raw_median", lambda x: int(x.isna().sum())))
print(by.to_string())
print("\nTask per subject from session_type agrees with target_modality of the trials (0 auditory, 1 visual):",
      bool((sessions.groupby("subject").modality.nunique() == 1).all()
           and ((sessions.session_type == "visual") == (sessions.modality == 1)).all()))
print("Training stages present:", sorted(sessions.stage.unique()))
print("Rigs:", sessions.rig.value_counts().to_dict())
print("\nTrials in the files:", len(alltrials), "| target trials:", len(trials_all),
      "| distractor trials:", int(sessions.n_distractor.sum()))
print("Outcome codes by stimulus type (0 target, 1 distractor):")
print(pd.crosstab(alltrials.stimulus_type, alltrials.outcome).to_string())
print("\nStimulus values (code's inpt) per task:")
print(trials_all.assign(task=np.where(trials_all.target_modality == 0, "auditory", "visual"))
      .groupby("task").inpt.apply(lambda x: sorted(np.round(x.unique(), 3))).to_string())
print("Visual difficulty codes present:", sorted(alltrials.visual_stim_difficulty.dropna().unique()),
      "| auditory difficulty values:", sorted(np.round(alltrials.auditory_stim_difficulty.dropna().unique(), 2)))
print("Trials table columns:", list(alltrials.columns))
has_ang = alltrials[alltrials.target_modality == 1].groupby("path").visual_gabor_angle.apply(lambda x: x.notna().any())
print(f"Visual-task sessions: {len(has_ang)}, of which {int(has_ang.sum())} have a visual_gabor_angle column")
vsets = trials_all[trials_all.target_modality == 1].groupby("path").inpt.apply(lambda x: tuple(np.round(np.unique(np.abs(x)), 2)))
print("Visual stimulus values |inpt| per session (get_stim_vals), sessions per set:", vsets.value_counts().to_dict())
print("Stimulus duration (ms):", alltrials.stimulus_duration.unique())
iti = alltrials.groupby("path").start_time.diff().dropna()
print(f"Start-to-start interval (s): median {iti.median():.2f}, 5-95% {iti.quantile(.05):.2f} to {iti.quantile(.95):.2f}")
print("\nSampling: face camera", sessions.face_rate.round(2).value_counts().to_dict(),
      "Hz; wheel", sessions.run_rate.round(1).value_counts().to_dict(), "Hz")
print("Raw running_speed per session (unit stated 'cm/s'): median of session medians "
      f"{sessions.run_raw_median.median():.3f}, median of session 99th percentiles {sessions.run_raw_p99.median():.3f}, "
      f"largest 99th percentile {sessions.run_raw_p99.max():.3f}")
print("Raw post hoc pupil diameter (pixels): session medians from "
      f"{sessions.pupil_raw_median.min():.1f} to {sessions.pupil_raw_median.max():.1f}")
cols = ["post_hoc_pupil_diameter", "face_energy", "running_speed", "post_hoc_pupil_std10", "face_std_10", "running_std10"]
print("\nMeasures at stimulus onset after the code's filtering and normalisation (target trials):")
print(trials_all[cols].describe().T[["count", "mean", "min", "50%", "max"]].round(3).to_string())
print("\nDuplicate recordings: identical face motion energy arrays", int(sessions.fp_face.duplicated(keep=False).sum()),
      "| identical trial tables (stimulus times and outcomes)", int(sessions.fp_trials.duplicated(keep=False).sum()))

The archive gives 13 subjects and 391 files; the files hold 13 mice, 8 on the auditory task and 5 on the visual task, all in training stages 5 and 6. No recording is deposited twice. `running_speed` is labelled cm/s, but its values (session medians around 0.07, 99th percentiles up to about 0.67) fit m/s, the unit of the paper (locomotion speed around 0.14 m/s) and of its code, which uses the values as they are; the values are taken as m/s and the label is recorded as a metadata error. Some sessions have a post hoc pupil series that is all missing; the paper's analyses that need pupil drop those trials, as here. The visual difficulty column is described as five angles, but only codes 0 to 2 occur. Most visual-task files also have a `visual_gabor_angle` column; for these, loadbehavior.get_stim_vals maps the codes 0, 1 and 2 to the session's distinct values of |angle - 45| / 45, largest first, and otherwise to 1, 0.8 and 0.6. The helper does the same, so the visual inputs are 1, 0.6 and 0.2 in most sessions (the sets per session are printed above).

The next cell checks that the measures share the clock of the trial table, with a contrast between conditions: if they do, the measures change after stimulus onset differently on trials with a response (a lick) than on trials without, and the difference shrinks when the same window is taken 4 s later, between trials.

In [ ]:
resp = trials_all.hmm_choice != 2
clock = []
for subj, g in trials_all.groupby("subject"):
    r = g.hmm_choice != 2
    row = {"subject": subj}
    for k in ["face_energy", "running_speed", "post_hoc_pupil_diameter"]:
        for w in ["d0", "d4"]:
            row[f"{k[:4]}_{w}"] = g.loc[r, f"{k}_{w}"].mean() - g.loc[~r, f"{k}_{w}"].mean()
    clock.append(row)
clock = pd.DataFrame(clock).set_index("subject")
print("Response minus no-response trials: change of each measure from stimulus onset to 1.5 s after (d0),")
print("and over the same 1.5 s starting 4 s after onset (d4). Face and pupil normalised units, running m/s.")
print(clock.round(4).to_string())
for k in ["face", "runn", "post"]:
    print(f"{k}: mice with a larger response contrast at onset than 4 s later (absolute values): "
          f"{int((clock[k + '_d0'].abs() > clock[k + '_d4'].abs()).sum())} of {len(clock)}; "
          f"median d0 {clock[k + '_d0'].median():.4f}, d4 {clock[k + '_d4'].median():.4f}")

## Sessions and trials (claim `cohort-inclusion`)

> Of 31 mice, 13 met the inclusion criteria (8 auditory, 5 visual task), with 381 of their 500 stage 5 and 6 sessions included.

The criterion is the STAR Methods' (Training and inclusion criteria): stage 5 or 6, at least 100 rewards with at least 20% at each port, and at least 10 such sessions per mouse. Rewards are counted from the `reward_left` and `reward_right` series; the hits of the trial table are shown as a second count. The sessions that pass are the sessions of every later analysis.

In [ ]:
s = sessions.copy()
s["rewards"] = s.reward_left + s.reward_right
s["min_port_frac"] = np.minimum(s.reward_left, s.reward_right) / s.rewards
hits = alltrials[alltrials.outcome == 1].groupby("path").target_port.agg(hits="size", hits_left=lambda x: int((x == 0).sum()))
s = s.merge(hits, left_on="path", right_index=True, how="left")
s["hit_min_frac"] = np.minimum(s.hits_left, s.hits - s.hits_left) / s.hits
s["pass_rewards"] = s.stage.isin(["S5", "S6"]) & (s.rewards >= 100) & (s.min_port_frac >= 0.2)
s["pass_hits"] = s.stage.isin(["S5", "S6"]) & (s.hits >= 100) & (s.hit_min_frac >= 0.2)
per_mouse = s.groupby("subject").pass_rewards.sum()
s["included"] = s.pass_rewards & s.subject.map(per_mouse >= 10)
print("Sessions in the dandiset:", len(s))
print("Pass by rewards:", int(s.pass_rewards.sum()), "| pass by hits:", int(s.pass_hits.sum()),
      "| either:", int((s.pass_rewards | s.pass_hits).sum()))
print("Mice with at least 10 passing sessions:", int((per_mouse >= 10).sum()), "of", len(per_mouse))
mods = s.groupby("subject").session_type.first()
print("Mice by task:", mods.value_counts().to_dict())
print("Included sessions (rewards):", int(s.included.sum()))
print("\nSessions that fail either count:")
print(s.loc[~s.pass_rewards | ~s.pass_hits, ["path", "stage", "rewards", "min_port_frac", "hits", "hit_min_frac"]]
      .round(3).to_string(index=False))
print("\nIncluded sessions per mouse:", s[s.included].groupby("subject").size().to_dict())
included = s[s.included]
trials = trials_all[trials_all.path.isin(included.path)].copy()
print("Target trials in included sessions:", len(trials))

**Result: Partially reproduced.** The dandiset holds 13 mice, 8 on the auditory and 5 on the visual task, all of which have at least 10 passing sessions, as the paper reports. Of its 391 sessions, 376 pass the criteria counted from the reward series (375 from the hits), not 381; the paper's 500 stage 5 and 6 sessions and its 18 excluded mice are not in the dandiset, so which 381 sessions were analysed cannot be told. The 376 sessions are used below.

## Training before stage 5 (claim `training-duration`)

> Mice that reached proficiency took 21 +/- 8 training sessions to reach stage 5.

The dandiset holds only stage 5 and 6 sessions (data check), so the sessions before stage 5 cannot be counted. Blocked: data missing.

## Distractor responses (claim `distractor-response-rate`)

> In stage 6, responses to distractors of the untrained modality were below 10%, with no discernible change in task performance.

Distractor trials have outcome 4 (`false_alarm`, a lick) or 8 (`correct_reject`, no lick); the rate is the fraction with a lick. All stage 6 sessions in the dandiset are used, and the included ones are shown too.

In [ ]:
d = alltrials[(alltrials.stimulus_type == 1)].merge(s[["path", "subject", "included"]], on="path")
d = d[d.stage == "S6"]
d["lick"] = (d.outcome == 4).astype(float)
for label, dd in [("all stage 6 sessions", d), ("included stage 6 sessions", d[d.included])]:
    pooled = dd.groupby("subject").lick.mean()
    per_sess = dd.groupby(["subject", "path"]).lick.mean().groupby("subject").mean()
    tab = pd.DataFrame({"sessions": dd.groupby("subject").path.nunique(), "distractor trials": dd.groupby("subject").size(),
                        "(a) pooled": pooled.round(4), "(b) mean over sessions": per_sess.round(4),
                        "max session": dd.groupby(["subject", "path"]).lick.mean().groupby("subject").max().round(4)})
    print(label)
    print(tab.to_string())
    print(f"(c) pooled over all mice: {dd.lick.mean():.4f} over {len(dd)} trials; mice below 0.10 under (a): "
          f"{int((pooled < 0.1).sum())} of {len(pooled)}, under (b): {int((per_sess < 0.1).sum())} of {len(per_sess)}\n")
st6 = trials_all.merge(s[["path", "included"]], on="path")
perf = st6[st6.hmm_choice != 2].assign(correct=lambda x: x.outcome == 1).groupby(["subject", "stage"]).correct.mean().unstack()
print("Not tested, for reference: fraction correct of responded target trials by stage")
print(perf.round(3).to_string())

**Result: Reproduced.** Under reading (c), 5.2% of the 47,037 stage 6 distractor trials had a lick (5.2% on the included sessions). Per mouse, 7 of the 8 mice with stage 6 sessions are below 10% under readings (a) and (b); BW053 is at 16.2% pooled (15.3% over sessions). The fraction correct of responded target trials changes between stages by at most 5.4 points (BW064, 0.816 to 0.762), shown but not tested.

## GLM-HMM: number of states (claim `figs2-n-states`)

> Cross-validated GLM-HMMs fit separately to each of the 13 mice gave between 3 and 5 states per mouse.

Per mouse, the target trials of the included sessions (distractor trials dropped, no-response trials kept) with inputs [stimulus value, 1] and choices left, right or none (claims file, M1 and M2). The GLM-HMM is the paper's ssm model, reimplemented in `hulsey.py` (its docstring gives the source of each part). 5-fold cross-validation over sessions, 1 to 7 states, MLE and MAP fits (2_n_states.py). 10 random initialisations per fold and number of states, as the paper. The number of states is the plateau rule of 3_generate_hmms.py on the test log likelihood per trial (mean over folds, best initialisation): reading (b) on the MLE curve alone, the code's; reading (a) the larger of the MLE and MAP plateaus.

In [ ]:
N_INIT_CV = 10
t0 = time.time()
subjects = sorted(trials.subject.unique())
seqs = {subj: hulsey.sequences(trials[trials.subject == subj]) for subj in subjects}
tasks, keys = [], []
for i, subj in enumerate(subjects):
    Xs, ys = seqs[subj]
    tt, kk = hulsey.cv_tasks(Xs, ys, N_INIT_CV, seed=1000 + i)
    tasks += tt
    keys += [(subj,) + k for k in kk]
order = np.argsort([-sum(len(x) for x in t[1]) * t[0] for t in tasks])  # largest fits first
res = hulsey.run_parallel(hulsey.cv_test_ll, [tasks[j] for j in order])
cv = {keys[j]: r for j, r in zip(order, res)}
print(f"{len(tasks)} cross-validation fits in {(time.time() - t0) / 60:.1f} min; "
      f"{sum(r[2] == 1000 for r in res)} stopped at 1000 iterations")
n_states, curves = {}, {}
rows = []
for subj in subjects:
    row = {"subject": subj, "task": mods[subj], "trials": sum(len(x) for x in seqs[subj][0]),
           "sessions": len(seqs[subj][0])}
    for use_map, name in [(False, "MLE"), (True, "MAP")]:
        a = np.full((N_INIT_CV, 7, 5), np.nan)
        for (sb, f, K, z, m), v in cv.items():
            if sb == subj and m == use_map:
                a[z, K - 1, f] = v[1]
        n, curve = hulsey.select_states(a)
        curves[(subj, name)] = curve
        row[name] = n
    row["(a) max"] = max(row["MLE"], row["MAP"])
    row["(b) MLE"] = row["MLE"]
    rows.append(row)
nst = pd.DataFrame(rows).set_index("subject")
print(nst.to_string())
for r in ["(a) max", "(b) MLE"]:
    print(f"reading {r}: range {nst[r].min()} to {nst[r].max()}; counts {nst[r].value_counts().sort_index().to_dict()}")
print("\nTest log likelihood per trial (MLE, mean over folds, best initialisation), 1 to 7 states:")
print(pd.DataFrame({subj: curves[(subj, 'MLE')] for subj in subjects}, index=range(1, 8)).T.round(4).to_string())
print("\nThe same for MAP:")
print(pd.DataFrame({subj: curves[(subj, 'MAP')] for subj in subjects}, index=range(1, 8)).T.round(4).to_string())
n_states = nst["(b) MLE"].to_dict()

**Result: Reproduced.** Under both readings every mouse has 3 or 4 states (five mice with 3, eight with 4), inside the claimed 3 to 5; no mouse has 5 states. The MAP plateau is never larger than the MLE plateau, so reading (a) gives the same counts as reading (b).

### Final models and state labels

As 3_generate_hmms.py: per mouse, 10 MLE fits on all sessions with the selected number of states (reading (b), which is also reading (a) when they agree), the one with the highest log likelihood kept; trials in a state when its posterior is at least 0.8 (flex_hmm.get_posterior_probs); states ordered optimal, disengaged, others (flex_hmm.permute_hmm). Sub-optimal states are then matched to the four templates of Figure2.py, on the sessions in which the optimal state occurs.

In [ ]:
t0 = time.time()
tasks, owner = [], []
for i, subj in enumerate(subjects):
    Xs, ys = seqs[subj]
    tt = hulsey.final_tasks(Xs, ys, n_states[subj], 10, seed=2000 + i)
    tasks += tt
    owner += [subj] * len(tt)
res = hulsey.run_parallel(hulsey.fit_one, tasks)
models = {}
for subj in subjects:
    rr = [res[j] for j in range(len(res)) if owner[j] == subj]
    models[subj] = rr[int(np.argmax([r[1] for r in rr]))][0]
print(f"final fits in {(time.time() - t0) / 60:.1f} min")
post_cols = []
parts = []
for subj in subjects:
    g = trials[trials.subject == subj]
    m = models[subj]
    p0 = hulsey.posterior_states(m, g)
    ps, _ = hulsey.psychos(g, p0.hmm_state.values)
    perm = hulsey.permutation_optimal_disengaged(ps)
    m.permute(perm)
    parts.append(hulsey.posterior_states(m, g))
post = pd.concat(parts)
trials = trials.drop(columns=[c for c in trials.columns if c.startswith("state_") or c == "hmm_state"], errors="ignore")
trials = trials.join(post)
# sessions in which the optimal state occurs (Figure2.py) and with more than 10 optimal trials (Figure3.py etc.)
opt_n = trials.groupby("path").hmm_state.apply(lambda x: int((x == 0).sum()))
trials["sess_has_opt"] = trials.path.map(opt_n > 0)
trials["sess_opt10"] = trials.path.map(opt_n > 10) & trials.path.map(included.set_index("path").pupil_raw_median.notna())
types = {}
rows = []
for subj in subjects:
    g = trials[(trials.subject == subj) & trials.sess_has_opt]
    ps, inp = hulsey.psychos(g, g.hmm_state.values)
    lab, sim = hulsey.state_types(ps)
    lab_all, _ = hulsey.state_types(ps, fixed_first_two=False)
    types[subj] = lab
    occ = g.hmm_state.value_counts(normalize=True).sort_index()
    rows.append({"subject": subj, "states": n_states[subj], "inputs": len(inp),
                 "labels": [hulsey.STATE_NAMES[k] for k in lab],
                 "nearest of all six templates": [hulsey.STATE_NAMES[k] for k in lab_all],
                 "occupancy": [round(occ.get(k, 0), 3) for k in range(n_states[subj])],
                 "indeterminate": round(g.hmm_state.isna().mean(), 3)})
print(pd.DataFrame(rows).set_index("subject").to_string())
trials["state_type"] = np.nan
for subj in subjects:
    sel = trials.subject == subj
    st = trials.loc[sel, "hmm_state"]
    trials.loc[sel, "state_type"] = st.map(lambda k: types[subj][int(k)] if k == k else np.nan)
print("\nSessions with the optimal state:", int(trials.groupby("path").sess_has_opt.first().sum()),
      "| with more than 10 optimal trials and a post hoc pupil (M5):", int(trials.groupby("path").sess_opt10.first().sum()),
      "of", trials.path.nunique())
m5 = trials[trials.sess_opt10]
print("Trials in the M5 sessions:", len(m5), "| with all six measures:", len(m5.dropna(subset=["post_hoc_pupil_diameter",
      "post_hoc_pupil_std10", "face_energy", "face_std_10", "running_speed", "running_std10"])),
      "| of these in a discrete state:", int(m5.dropna(subset=["post_hoc_pupil_diameter", "post_hoc_pupil_std10", "face_energy",
      "face_std_10", "running_speed", "running_std10"]).hmm_state.notna().sum()))
print("\nTrials by state (0 optimal, 1 disengaged, 2 or more other; NaN indeterminate):",
      trials.hmm_state.value_counts(dropna=False).sort_index().to_dict())

In [ ]:
# a state with no discrete trial in the sessions where the optimal state occurs is typed on all sessions
for subj in subjects:
    if (types[subj] == -1).any():
        g = trials[trials.subject == subj]
        ps, _ = hulsey.psychos(g, g.hmm_state.values)
        lab_all_sess, _ = hulsey.state_types(ps)
        print(f"{subj}: states with no discrete trial in sessions with the optimal state:",
              list(np.where(types[subj] == -1)[0]), "| typed on all sessions:",
              [hulsey.STATE_NAMES[k] for k in lab_all_sess],
              "| trials per state over all sessions:", g.hmm_state.value_counts().sort_index().to_dict())
        types[subj] = np.where(types[subj] == -1, lab_all_sess, types[subj])
        sel = trials.subject == subj
        trials.loc[sel, "state_type"] = trials.loc[sel, "hmm_state"].map(lambda k: types[subj][int(k)] if k == k else np.nan)

## State types (claim `fig2a-state-types`)

> Six stereotyped performance states emerged across mice. Every mouse had an optimal and a disengaged state and at least one sub-optimal state (bias left or right, avoid left or right).

In [ ]:
rows = []
for subj in subjects:
    lab = types[subj]
    rows.append({"subject": subj, "states": len(lab), "optimal": int((lab == 0).sum()), "disengaged": int((lab == 1).sum()),
                 "other states": int((lab >= 2).sum()),
                 "other states matched to a sub-optimal template": int(((lab >= 2) & (lab <= 5)).sum())})
ft = pd.DataFrame(rows).set_index("subject")
print(ft.to_string())
ok = (ft.optimal == 1) & (ft.disengaged == 1) & (ft["other states"] >= 1) & (ft["other states"] == ft["other states matched to a sub-optimal template"])
print("\nMice with an optimal, a disengaged and at least one other state, each other state typed:", int(ok.sum()), "of", len(ft))
counts = {hulsey.STATE_NAMES[k]: int(sum((types[s] == k).any() for s in subjects)) for k in range(2, 6)}
print("Mice with each sub-optimal type:", counts, "(Figure 2A: bias left 6, avoid right 4, bias right 5, avoid left 5)")
print("Sub-optimal states in all:", int(sum((types[s] >= 2).sum() for s in subjects)))
print("\nDistance of each state's psychometric (sessions with the optimal state) to the six templates of Figure2.py;"
      " the smallest per row is the nearest template:")
for subj in ["BW061", "BW048"]:
    g = trials[(trials.subject == subj) & trials.sess_has_opt]
    ps, inp = hulsey.psychos(g, g.hmm_state.values)
    _, sim = hulsey.state_types(ps, fixed_first_two=False)
    print(f"{subj}, labels {[hulsey.STATE_NAMES[k] for k in types[subj]]}:")
    print(pd.DataFrame(sim, columns=hulsey.STATE_NAMES[:6]).round(2).to_string())
    for k in range(len(ps)):
        print(f"  state {k} ({hulsey.STATE_NAMES[types[subj][k]]}), P(left), P(right), P(no response) by input {np.round(inp, 2).tolist()}:")
        print("   ", np.round(ps[k], 2).tolist())


**Result: Reproduced.** All 13 mice have an optimal state, a disengaged state and at least one other state, and every other state matches a sub-optimal template (21 sub-optimal states). The mice per sub-optimal type are bias left 6, avoid right 5, bias right 6 and avoid left 4, against 6, 4, 5 and 5 in Figure 2A; these counts do not decide the claim. In BW059 one state has no trial with a posterior of at least 0.8 in the sessions where the optimal state occurs; it is typed on all sessions (avoid right), an assumption. Under M3 each other state is matched only to the four sub-optimal templates, so the matching itself cannot fail. The distances to all six templates printed for BW061 and BW048 show each of their states nearest the template of its label (for example BW048 state 2, avoid left, 2.96 against 4.68 for the next nearest).

## Occupancy (claim `fig2b-optimal-occupancy`)

> Mice spent the largest proportion of trials in the optimal state (around half), more than in the disengaged, sub-optimal or indeterminate states; 13.1% +/- 1.6% of trials were indeterminate.

Per mouse, trials of the sessions in which the optimal state occurs, pooled (Figure2.py).

In [ ]:
occ = []
for subj in subjects:
    g = trials[(trials.subject == subj) & trials.sess_has_opt]
    h = g.hmm_state
    occ.append({"subject": subj, "optimal": (h == 0).mean(), "disengaged": (h == 1).mean(),
                "sub-optimal": (h > 1).mean(), "indeterminate": h.isna().mean(), "trials": len(g)})
occ = pd.DataFrame(occ).set_index("subject")
print((occ.drop(columns="trials") * 100).round(1).assign(trials=occ.trials).to_string())
m_ = occ.drop(columns="trials").mean() * 100
se_ = occ.drop(columns="trials").sem() * 100
print("\nmean +/- SEM over mice (%):", {k: f"{m_[k]:.1f} +/- {se_[k]:.1f}" for k in m_.index})
for other in ["disengaged", "sub-optimal", "indeterminate"]:
    w = stats.wilcoxon(occ.optimal, occ[other])
    print(f"optimal vs {other}: optimal larger in {int((occ.optimal > occ[other]).sum())} of 13 mice, "
          f"Wilcoxon signed-rank p = {w.pvalue:.2g}")

**Result: Reproduced.** The optimal state holds the largest share of trials (50.5% +/- 3.3%), more than the disengaged (12.7%), sub-optimal (23.7%) and indeterminate (13.1%) states, larger in 13 of 13 mice for each (Wilcoxon signed-rank p = 0.00024 for all three). The indeterminate share, 13.1% +/- 1.6%, matches the expected 13.1% +/- 1.6%.

## Dwell times (claim `fig2c-dwell-times`)

> Epochs of the optimal state lasted longer than disengaged and sub-optimal epochs (about 37 against about 15 trials), and the median transition between discrete states took 3.4 +/- 0.3 trials.

Epochs from flex_hmm.get_dwell_times on the sessions in which the optimal state occurs (Figure2.py), with (y) the code's dwell and (x) the run length. A disengaged epoch longer than 40 trials (code's dwell) that ends a session is left out (Figure2.py). The per-mouse median is the first length at which the cumulative distribution exceeds 0.499. get_dwell_times also adds a last epoch of dwell 0 when a session's last trial differs from the one before it, which adds a zero to the indeterminate dwells; the code's output is used, and a variant without these epochs is shown.

In [ ]:
def cdf_median(v):
    v = np.asarray(v)
    for i in range(500):
        if np.mean(v <= i) > 0.499:
            return i
    return np.nan

ep_rows, ind_rows = [], []
for subj in subjects:
    g = trials[(trials.subject == subj) & trials.sess_has_opt].sort_values(["path", "trial_in_session"])
    for path, sg in g.groupby("path"):
        dt = hulsey.dwell_table(sg.hmm_state.values)
        if len(dt) == 0:
            continue
        gap = dt.first_trial.values - np.r_[0, dt.last_trial.values[:-1]]
        ind_rows += [{"subject": subj, "gap": x} for x in gap]
        drop_last = dt.state.iloc[-1] == 1 and dt.dwell.iloc[-1] > 40
        dt = dt.iloc[:-1] if drop_last else dt
        ep_rows.append(dt.assign(subject=subj, path=path))
ep = pd.concat(ep_rows, ignore_index=True)
ep["group"] = np.where(ep.state == 0, "optimal", np.where(ep.state == 1, "disengaged", "sub-optimal"))
ind = pd.DataFrame(ind_rows)
res_dw = {}
for col, cname in [("dwell", "(y) code dwell"), ("run", "(x) run length")]:
    med = ep.groupby(["subject", "group"])[col].apply(cdf_median).unstack()
    mean = ep.groupby(["subject", "group"])[col].mean().unstack()
    pooled = ep.groupby("group")[col].mean()
    res_dw[cname] = (med, mean, pooled)
    print(cname)
    print("  (a) mean over mice of per-mouse median:", med.mean().round(1).to_dict())
    print("  (b) mean over mice of per-mouse mean:  ", mean.mean().round(1).to_dict())
    print("  (c) mean over all epochs pooled:       ", pooled.round(1).to_dict())
    for other in ["disengaged", "sub-optimal"]:
        for nm, tab in [("medians", med), ("means", mean)]:
            w = stats.wilcoxon(tab.optimal, tab[other])
            print(f"  optimal vs {other} ({nm}): optimal longer in {int((tab.optimal > tab[other]).sum())} of 13, p = {w.pvalue:.2g}")
ind_med = ind.groupby("subject").gap.apply(cdf_median)
print("\nMedian indeterminate dwell per mouse:", ind_med.to_dict())
print(f"mean +/- SEM over mice: {ind_med.mean():.2f} +/- {ind_med.sem():.2f} trials")
print("Per-mouse medians (code dwell):")
print(res_dw["(y) code dwell"][0].assign(indeterminate=ind_med).to_string())
print("Epochs per group:", ep.group.value_counts().to_dict())
# get_dwell_times adds a last epoch of dwell 0 (run 1) when a session's last trial differs from the one before
# it; it also adds a zero gap to the indeterminate dwells. Variant without these epochs and gaps:
spur = ep.dwell == 0
print(f"\nEpochs of dwell 0 (the extra last epoch): {int(spur.sum())} of {len(ep)}")
ind2 = []
for subj in subjects:
    g = trials[(trials.subject == subj) & trials.sess_has_opt].sort_values(["path", "trial_in_session"])
    for path, sg in g.groupby("path"):
        dt = hulsey.dwell_table(sg.hmm_state.values)
        dt = dt[dt.dwell > 0]
        if len(dt) == 0:
            continue
        gap = dt.first_trial.values - np.r_[0, dt.last_trial.values[:-1]]
        ind2 += [{"subject": subj, "gap": x} for x in gap]
ind_med2 = pd.DataFrame(ind2).groupby("subject").gap.apply(cdf_median)
print(f"Without them: median indeterminate dwell {ind_med2.mean():.2f} +/- {ind_med2.sem():.2f} trials")
ep2 = ep[~spur]
for col, cname in [("dwell", "(y) code dwell"), ("run", "(x) run length")]:
    med2 = ep2.groupby(["subject", "group"])[col].apply(cdf_median).unstack()
    print(f"  {cname}, (a) mean over mice of per-mouse median:", med2.mean().round(1).to_dict())

**Result: Partially reproduced.** Optimal epochs last longer than disengaged and sub-optimal epochs in 13 of 13 mice under every reading (p = 0.00024), and the median indeterminate dwell, 3.31 +/- 0.35 trials, is within 3.4 +/- 0.3 (3.31 also without the 3 extra epochs of dwell 0). The dwells are longer than the paper's 37 and about 15 under every reading: reading (a) gives 43.8 trials optimal, 14.0 disengaged and 19.1 sub-optimal with the code dwell, and 42.8, 13.8 and 18.1 as run length; the optimal dwell is above 40.7 and the sub-optimal above 16.5 in both, and readings (b) and (c) give 49 to 74 trials for the optimal state.

## Choice prediction (claims `fig2d-prediction-accuracy` and `fig1g-example-mouse-prediction`)

> The best-fit GLM-HMM predicted 75.7% +/- 1.7% of choices, a 10.8% +/- 1.3% improvement over a one-state (static performance) model.

As 6_predictive_accuracy.py: per mouse, 5 folds, each holding out a contiguous block of 20% of every session's trials, the order of the blocks permuted per session; on the training trials, concatenated as one sequence, a one-state model and a model with the mouse's number of states are fit (flex_hmm.choice_hmm_fit, MAP), 5 initialisations each, the best by test log likelihood kept; each test trial's choice is predicted from the state the transition matrix makes most likely after the filtered state of the preceding test trials.

In [ ]:
rng_pred = np.random.default_rng(6)
t0 = time.time()
tasks, meta = [], []
folds = {}
for subj in subjects:
    g = trials[trials.subject == subj].sort_values(["path", "trial_in_session"])
    X = np.c_[g.inpt.values, np.ones(len(g))]
    y = g.hmm_choice.values.astype(int)
    sess_idx = pd.factorize(g.path)[0]
    tr_i, te_i = hulsey.stratified_kfold_random_order(sess_idx, 5, rng_pred)
    folds[subj] = (X, y, tr_i, te_i)
    for f in range(5):
        for K in (1, n_states[subj]):
            for z in range(5):
                tasks.append((K, X[tr_i[f]], y[tr_i[f]], X[te_i[f]], y[te_i[f]], int(rng_pred.integers(2**31))))
                meta.append((subj, f, K))
res = hulsey.run_parallel(hulsey.fit_map_one, tasks)
print(f"{len(tasks)} fits in {(time.time() - t0) / 60:.1f} min")
acc = {}
for subj in subjects:
    X, y, tr_i, te_i = folds[subj]
    for K in (1, n_states[subj]):
        a = []
        for f in range(5):
            cand = [res[j] for j in range(len(res)) if meta[j] == (subj, f, K)]
            best = cand[int(np.argmax([c[1] for c in cand]))][0]
            pred = hulsey.predict_choices(best, X[te_i[f]], y[te_i[f]])
            a.append(np.mean(pred == y[te_i[f]]))
        acc[(subj, "one state" if K == 1 else "best")] = np.mean(a) * 100
acc = pd.Series(acc).unstack()
acc["states"] = pd.Series(n_states)
acc["gain (points)"] = acc.best - acc["one state"]
acc["gain (relative %)"] = (acc.best / acc["one state"] - 1) * 100
print(acc.round(2).to_string())
print(f"\nbest model: {acc.best.mean():.1f} +/- {acc.best.sem():.1f}%; one state: {acc['one state'].mean():.1f} +/- {acc['one state'].sem():.1f}%")
print(f"(a) gain in points: {acc['gain (points)'].mean():.1f} +/- {acc['gain (points)'].sem():.1f}; "
      f"(b) relative gain: {acc['gain (relative %)'].mean():.1f} +/- {acc['gain (relative %)'].sem():.1f}%")
print(f"best better in {int((acc['gain (points)'] > 0).sum())} of 13; Wilcoxon signed-rank p = {stats.wilcoxon(acc.best, acc['one state']).pvalue:.2g}; "
      f"rank-sum (variant, O4) p = {stats.ranksums(acc.best, acc['one state']).pvalue:.2g}")

**Result: Reproduced.** The selected GLM-HMM predicts 76.2% +/- 1.6% of choices against 65.1% for one state, a gain of 11.0 +/- 1.3 points (17.8% relative), better in 13 of 13 mice (signed-rank p = 0.00024; rank-sum p = 0.00025).

> For the example mouse, a 4-state GLM-HMM was selected; it predicted 78% of choices against 70% for a one-state model.

The example mouse is BW046 (Figure1.py, 7_example_mouse_decoding.py).

In [ ]:
print("BW046: states selected", n_states["BW046"], "(reading (a):", nst.loc["BW046", "(a) max"], ")")
print(f"BW046: one state {acc.loc['BW046', 'one state']:.1f}%, best model {acc.loc['BW046', 'best']:.1f}%")

**Result: Reproduced.** BW046 has 4 states under both readings, and its choices are predicted at 78.1% by the 4-state model against 70.2% by the one-state model.

## Pupil diameter and the optimal state (claims `fig3c-inverted-u-pupil`, `fig3d-optimal-pupil-modality`, `fig3e-u-disengaged-pupil`)

> The probability of being in the optimal state follows an inverted-U function of pupil diameter, highest at intermediate diameters, in nearly all mice.

Sessions with more than 10 optimal-state trials and a post hoc pupil (claims file, M5). As 4_optimal_pupils.py: discrete-state trials with a pupil value are binned by pupil diameter in 0.02 bins (lower, upper]; per fold of a shuffled 5-fold split, P(optimal) per bin with more than 2 training or 3 test trials, polynomials of degree 1 to 7 fit to the training bins and scored by r2 on the test bins; the degree is the elbow of the mean r2 (largest height above the line from degree 1 to degree 7). As Figure3.py: P(state) per 0.02 bin [lower, upper) over trials with all six measures, for bins with more than 5 trials (a state absent from a bin is left missing, as the code does), the bins below the first one with a value merged into it; the polynomial of the selected degree is fit to P(optimal) and the optimal pupil is its maximum on a 0.001 grid within the range of the bins.

In [ ]:
MEAS6 = ["post_hoc_pupil_diameter", "post_hoc_pupil_std10", "face_energy", "face_std_10", "running_speed", "running_std10"]

def fig3_set(subj, sess_col="sess_opt10"):
    """Figure3.py / FigureS3_S4.py / FigureS8.py trials: the mouse's sessions in the set, all six
    measures present, the movement index z-scored over these trials."""
    g = trials[(trials.subject == subj) & trials[sess_col]].dropna(subset=MEAS6).copy()
    g["movement"] = stats.zscore(g.face_energy) + stats.zscore(g.running_speed)
    g["movement_std"] = stats.zscore(g.face_std_10) + stats.zscore(g.running_std10)
    return g

def p_state_bins(values, state, lo, hi, step, min_n=5, merge_low=True):
    """Figure3.py: P(optimal), P(disengaged), P(other) per bin [p, p + step) with more than min_n
    trials; a state absent from a bin stays NaN; bins below the first with an optimal value are merged into it."""
    st = np.where(state > 2, 2, state)
    edges = np.arange(lo, hi, step)
    out = np.full((len(edges), 3), np.nan)
    for k, p in enumerate(edges):
        sel = (values >= p) & (values < p + step)
        n = sel.sum()
        if n > min_n:
            for s_ in np.unique(st[sel]):
                out[k, int(s_)] = np.mean(st[sel] == s_)
    if merge_low and np.isfinite(out[:, 0]).any():
        k = np.where(np.isfinite(out[:, 0]))[0][0]
        sel = (values >= lo) & (values < lo + (k + 1) * step)
        if sel.sum() > min_n:
            for s_ in np.unique(st[sel]):
                out[k, int(s_)] = np.mean(st[sel] == s_)
    return edges, out

def cv_r2(values, opt, edges_lower, step, rng, degrees=range(1, 8)):
    """4_optimal_pupils.py: 5-fold shuffled split of trials; binned P(optimal) in (lower, upper]
    with more than 2 training and 3 test trials per bin; test r2 of polynomial fits per degree."""
    from sklearn.model_selection import KFold
    from sklearn.metrics import r2_score
    kf = KFold(n_splits=5, shuffle=True, random_state=int(rng.integers(2**31)))
    r2 = np.full((len(degrees), 5), np.nan)
    def binp(v, o, mn):
        y = np.full(len(edges_lower), np.nan)
        for i, lo in enumerate(edges_lower):
            sel = (v > lo) & (v <= lo + step)
            if sel.sum() > mn:
                y[i] = o[sel].mean()
        return y
    for f, (a, b) in enumerate(kf.split(values)):
        ytr, yte = binp(values[a], opt[a], 2), binp(values[b], opt[b], 3)
        x = np.asarray(edges_lower, dtype=np.float32)
        xtr, ytr2 = x[np.isfinite(ytr)], ytr[np.isfinite(ytr)]
        xte, yte2 = x[np.isfinite(yte)], yte[np.isfinite(yte)]
        for j, d in enumerate(degrees):
            c = np.polyfit(xtr, ytr2, d)
            r2[j, f] = r2_score(yte2, np.poly1d(c)(xte))
    return r2.mean(axis=1)

def elbow_degree(mean_r2):
    """4_optimal_pupils.py: the degree (2 to 6) whose mean r2 lies furthest above the line from degree 1 to 7."""
    x1, y1, x2, y2 = 0, mean_r2[0], 6, mean_r2[6]
    b = (x1 * y2 - x2 * y1) / (x1 - x2)
    m = (y1 - y2) / (x1 - x2)
    signs = [mean_r2[ii] - (m * ii + b) for ii in range(1, 6)]
    return int(np.argmax(signs) + 2)

rng3 = np.random.default_rng(3)
pupil_rows, curves_pupil = [], {}
for subj in subjects:
    g4 = trials[(trials.subject == subj) & trials.sess_opt10].dropna(subset=["post_hoc_pupil_diameter", "hmm_state"])
    r2 = cv_r2(g4.post_hoc_pupil_diameter.values, (g4.hmm_state.values == 0).astype(float), np.arange(0, 1, .02), .02, rng3)
    deg = elbow_degree(r2)
    g = fig3_set(subj).dropna(subset=["hmm_state"])
    edges, ps3 = p_state_bins(g.post_hoc_pupil_diameter.values, g.hmm_state.values, 0, 1, .02)
    curves_pupil[subj] = (edges, ps3)
    y = ps3[:, 0]
    x2, y2 = edges[np.isfinite(y)], y[np.isfinite(y)]
    xs = np.arange(x2[0], x2[-1], .001)
    fit = np.poly1d(np.polyfit(x2.astype(np.float32), y2.astype(np.float32), deg))(xs)
    opt_pupil = xs[np.argmax(fit)]
    c2 = np.polyfit(x2, y2, 2)
    pupil_rows.append({"subject": subj, "task": mods[subj], "trials": len(g), "bins": len(x2),
                       "range": f"{x2[0]:.2f}-{x2[-1]:.2f}", "degree": deg, "r2 deg2": r2[1],
                       "optimal pupil": opt_pupil, "fit at max": fit.max(), "fit at low end": fit[0], "fit at high end": fit[-1],
                       "(i) interior max, lower ends": bool(0 < np.argmax(fit) < len(fit) - 1 and fit[0] < fit.max() and fit[-1] < fit.max()),
                       "(ii) quad coef": c2[0]})
pupil_tab = pd.DataFrame(pupil_rows).set_index("subject")
print(pupil_tab.round(3).to_string())
n_i = int(pupil_tab["(i) interior max, lower ends"].sum())
n_ii = int((pupil_tab["(ii) quad coef"] < 0).sum())
print(f"\n(i) maximum inside the range with lower values at both ends: {n_i} of 13; (ii) negative quadratic term: {n_ii} of 13")
print("Readings of 'nearly all': (a) at least 12 of 13, (b) at least 11 of 13:",
      {"(i)": (n_i >= 12, n_i >= 11), "(ii)": (n_ii >= 12, n_ii >= 11)})
optimal_pupil = pupil_tab["optimal pupil"]

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(18, 5), sharey=True)
for ax, subj in zip(axes.ravel(), subjects):
    edges, ps3 = curves_pupil[subj]
    for k, c in zip(range(3), ["teal", "firebrick", "orange"]):
        ax.plot(edges, ps3[:, k], ".-", color=c, lw=1)
    ax.axvline(optimal_pupil[subj], color="k", ls=":")
    ax.set_title(subj, fontsize=9)
    ax.set_xlim(0, 1)
axes.ravel()[-1].axis("off")
fig.supxlabel("pupil diameter (fraction of session maximum)")
fig.supylabel("P(state): optimal teal, disengaged red, other orange")
plt.tight_layout()
plt.show()
print("All plotted values lie in [0, 1] on both axes; dotted line: optimal pupil.")

**Result: Reproduced.** In 13 of 13 mice the fit of P(optimal) on pupil diameter has its maximum inside the data range with lower values at both ends, and in 13 of 13 a second-order fit has a negative quadratic term, so "nearly all" holds under every reading.

> The optimal pupil diameter is smaller in mice performing the auditory task than in those performing the visual task.

In [ ]:
aud = optimal_pupil[pupil_tab.task == "auditory"]
vis = optimal_pupil[pupil_tab.task == "visual"]
rs = stats.ranksums(aud, vis)
print("auditory:", aud.round(3).to_dict())
print("visual:", vis.round(3).to_dict())
print(f"median auditory {aud.median():.3f}, visual {vis.median():.3f}; Wilcoxon rank-sum p = {rs.pvalue:.3g}; "
      f"Mann-Whitney U exact p = {stats.mannwhitneyu(aud, vis).pvalue:.3g}")

**Result: Reproduced.** The optimal pupil is smaller in auditory-task mice (median 0.649 of the session maximum) than in visual-task mice (0.739); Wilcoxon rank-sum p = 0.013 (exact Mann-Whitney p = 0.011).

> Across mice, aligned to each mouse's optimal pupil, the probability of the optimal state is an inverted U and the probability of the disengaged state a U of pupil diameter; this holds also on trials when the mice were stationary.

As Figure3.py: each mouse's P(state) per 0.02 bin shifted so that the bin nearest its optimal pupil is at 0, and averaged over mice. The shape test is the claims file's: on bins with data from at least 7 of 13 mice, a second-order fit with a negative (optimal) or positive (disengaged) quadratic term and its vertex inside the range. Stationary trials, readings (a) locomotion speed exactly 0 and (b) absolute speed below 0.02 m/s, with the same optimal pupils; (b') speed below 0.02 m/s with backward movement counted as stationary, the rule of the first version of this notebook, is shown for comparison.

In [ ]:
def shifted_mean(sel_fn):
    shifted = np.full((len(subjects), 3, 50), np.nan)
    x = np.arange(0, 1, .02)
    for i, subj in enumerate(subjects):
        g = fig3_set(subj).dropna(subset=["hmm_state"])
        g = g[sel_fn(g)]
        if len(g) == 0:
            continue
        _, ps3 = p_state_bins(g.post_hoc_pupil_diameter.values, g.hmm_state.values, 0, 1, .02)
        ob = np.argmin(np.abs(x - optimal_pupil[subj]))
        for j, b in enumerate(range(ob - 25, ob + 25)):
            if 0 <= b < 50:
                shifted[i, :, j] = ps3[b]
    return shifted

def shape(curve_by_mouse, k, sign):
    n = np.sum(np.isfinite(curve_by_mouse[:, k, :]), axis=0)
    mean = np.nanmean(curve_by_mouse[:, k, :], axis=0)
    keep = n >= 7
    xb = np.arange(-50, 50, 2)[keep]
    if keep.sum() < 3:
        return dict(bins=int(keep.sum()), quad=np.nan, vertex=np.nan, holds=False)
    c = np.polyfit(xb, mean[keep], 2)
    vx = -c[1] / (2 * c[0])
    return dict(bins=int(keep.sum()), range=f"{xb[0]} to {xb[-1]}", quad=c[0], vertex=round(vx, 1),
                holds=bool(np.sign(c[0]) == sign and xb[0] < vx < xb[-1]))

shift_res = {}
for name, fn in [("all trials", lambda g: np.ones(len(g), bool)),
                 ("(a) stationary, speed == 0", lambda g: g.running_speed.values == 0),
                 ("(b) stationary, |speed| < 0.02 m/s", lambda g: np.abs(g.running_speed.values) < 0.02),
                 ("(b') speed < 0.02 m/s, backward included", lambda g: g.running_speed.values < 0.02)]:
    sh = shifted_mean(fn)
    shift_res[name] = sh
    o, d = shape(sh, 0, -1), shape(sh, 1, +1)
    nt = sum(int(fn(fig3_set(s_).dropna(subset=["hmm_state"])).sum()) for s_ in subjects)
    print(f"{name} ({nt} trials): optimal {o}\n    disengaged {d}")
nback = sum(int((fig3_set(s_).dropna(subset=["hmm_state"]).running_speed.values <= -0.02).sum()) for s_ in subjects)
print(f"Trials with speed at or below -0.02 m/s (backward), counted as stationary by (b') only: {nback}")
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5), sharey=True)
for a, (name, sh) in zip(ax, shift_res.items()):
    for k, c in zip(range(3), ["teal", "firebrick", "orange"]):
        a.plot(np.arange(-50, 50, 2), np.nanmean(sh[:, k, :], axis=0), color=c)
    a.set_title(name, fontsize=9)
    a.set_xlabel("pupil minus optimal pupil (% of max)")
ax[0].set_ylabel("mean P(state) over mice")
plt.tight_layout()
plt.show()
print("x from -50 to 48 % of max; y in [0, 1].")

**Result: Reproduced.** Aligned to each mouse's optimal pupil and averaged, P(optimal) has a negative and P(disengaged) a positive quadratic term with the vertex inside the range (23 bins with at least 7 mice), and both shapes hold on stationary trials with absolute speed below 0.02 m/s (reading (b), 32,495 trials, 21 bins; vertices -3.3 and -6.1). Counting the 214 backward trials at or below -0.02 m/s as stationary (b') changes the vertices by at most 0.1. Reading (a), speed exactly 0, leaves 1,821 trials and no bin with 7 mice, so the shapes cannot be tested under it.

## Movement measures and the optimal state (claims `figs4-r2-individual-movement`, `fig3f-movement-index-inverted-u`, `fig3f-movement-index-u-disengaged`, `figs5-movement-index-correlations`)

> Second-order cross-validated regression fits of optimal-state probability are poorer for face motion energy (r2 0.49 +/- 0.08) and locomotion speed (0.44 +/- 0.07) than for pupil diameter (0.72 +/- 0.03).

The cross-validated r2 of a degree-2 fit as 4_optimal_pupils.py computes it (above), for pupil on its own trial set, and for face motion energy, locomotion speed and the movement index on the Figure3.py trial set (sessions of M5, six measures present). Bins, O5: (a) the plotting bins of FigureS3_S4.py and Figure3.py (face 0.05 from 0 to 1, locomotion 0.02 m/s from -0.02 to 0.30, movement index 0.25 from -5 to 5); (b) 50 equal-width bins over each mouse's range.

In [ ]:
rng4 = np.random.default_rng(4)
BINS_A = {"face_energy": (0, 1, .05), "running_speed": (-.02, .301, .02), "movement": (-5, 5, .25)}
r2rows = []
for subj in subjects:
    g = fig3_set(subj).dropna(subset=["hmm_state"])
    opt = (g.hmm_state.values == 0).astype(float)
    row = {"subject": subj, "pupil": pupil_tab.loc[subj, "r2 deg2"]}
    for meas, (lo, hi, step) in BINS_A.items():
        v = g[meas].values
        row[f"{meas} (a)"] = cv_r2(v, opt, np.arange(lo, hi, step), step, rng4, degrees=[1, 2])[1]
        stepb = (v.max() - v.min()) / 50
        row[f"{meas} (b)"] = cv_r2(v, opt, v.min() - 1e-9 + np.arange(50) * stepb, stepb, rng4, degrees=[1, 2])[1]
    r2rows.append(row)
r2tab = pd.DataFrame(r2rows).set_index("subject")
print(r2tab.round(3).to_string())
print("\nmean +/- SEM over mice:", {c: f"{r2tab[c].mean():.4f} +/- {r2tab[c].sem():.4f}" for c in r2tab.columns})
for c in r2tab.columns[1:]:
    w = stats.wilcoxon(r2tab[c], r2tab.pupil)
    print(f"{c} vs pupil: difference {np.mean(r2tab[c] - r2tab.pupil):+.2f}, lower in {int((r2tab[c] < r2tab.pupil).sum())} of 13, p = {w.pvalue:.3g}")

In [ ]:
# Variant: the r2 above comes from one shuffled 5-fold split per mouse, as in 4_optimal_pupils.py.
# Mean over 10 splits per mouse, to show how much the split moves it.
rng4b = np.random.default_rng(41)
N_SPLITS = 10
rep_rows = []
for subj in subjects:
    g4 = trials[(trials.subject == subj) & trials.sess_opt10].dropna(subset=["post_hoc_pupil_diameter", "hmm_state"])
    o4 = (g4.hmm_state.values == 0).astype(float)
    pr = [cv_r2(g4.post_hoc_pupil_diameter.values, o4, np.arange(0, 1, .02), .02, rng4b, degrees=[1, 2])[1] for _ in range(N_SPLITS)]
    row = {"subject": subj, "pupil": np.mean(pr), "pupil min": np.min(pr), "pupil max": np.max(pr)}
    g = fig3_set(subj).dropna(subset=["hmm_state"])
    opt = (g.hmm_state.values == 0).astype(float)
    for meas, (lo, hi, step) in BINS_A.items():
        v = g[meas].values
        row[f"{meas} (a)"] = np.mean([cv_r2(v, opt, np.arange(lo, hi, step), step, rng4b, degrees=[1, 2])[1] for _ in range(N_SPLITS)])
        stepb = (v.max() - v.min()) / 50
        row[f"{meas} (b)"] = np.mean([cv_r2(v, opt, v.min() - 1e-9 + np.arange(50) * stepb, stepb, rng4b, degrees=[1, 2])[1] for _ in range(N_SPLITS)])
    rep_rows.append(row)
r2rep = pd.DataFrame(rep_rows).set_index("subject")
print(f"Mean r2 over {N_SPLITS} shuffled splits per mouse (pupil min and max over the splits):")
print(r2rep.round(3).to_string())
print("\nmean +/- SEM over mice:", {c: f"{r2rep[c].mean():.4f} +/- {r2rep[c].sem():.4f}" for c in r2rep.columns if c not in ("pupil min", "pupil max")})
for c in [c for c in r2rep.columns if "(" in c]:
    w = stats.wilcoxon(r2rep[c], r2rep.pupil)
    print(f"{c} vs pupil: difference {np.mean(r2rep[c] - r2rep.pupil):+.2f}, lower in {int((r2rep[c] < r2rep.pupil).sum())} of 13, p = {w.pvalue:.3g}")


**Result: Partially reproduced.** Pupil diameter gives 0.690 +/- 0.039, within 0.72 +/- 0.03. Face motion energy gives 0.428 +/- 0.092 with the plotting bins (a), within tolerance, and 0.409 with 50 bins (b), just outside; locomotion speed gives 0.567 under (a), outside, and 0.413 under (b), within. So no bin reading puts all three values in tolerance. Both measures are below pupil on average, but face is not lower at p < 0.05 under either reading (p = 0.11 and 0.057), and locomotion only under (b) (p = 0.011; (a) p = 0.41). Over 10 shuffled splits per mouse, pupil gives 0.682 +/- 0.043 (BW051 from 0.001 to 0.562), and the tests come out the same way (face p = 0.15 and 0.068, locomotion p = 0.34 and 0.013); the single-split r2 of a mouse can move by more than 0.1 with the split.

> Optimal-state probability peaks at intermediate values of the movement index, an inverted U similar to pupil diameter (cross-validated r2 0.63 +/- 0.06, not different from pupil).

The movement index is the sum of the z-scored face motion energy and locomotion speed over the trials of the Figure3.py set (M4). Its r2 and the test against pupil are in the table above (rows `movement`). The shape is tested on the mean P(optimal) curve over the 0.25 bins of Figure3.py, as for pupil.

In [ ]:
mv = np.full((len(subjects), 3, 40), np.nan)
for i, subj in enumerate(subjects):
    g = fig3_set(subj).dropna(subset=["hmm_state"])
    _, ps3 = p_state_bins(g.movement.values, g.hmm_state.values, -5, 5, .25, merge_low=False)
    mv[i] = ps3.T

def shape_x(curves, k, sign, xb_all):
    n = np.sum(np.isfinite(curves[:, k, :]), axis=0)
    mean = np.nanmean(curves[:, k, :], axis=0)
    keep = n >= 7
    xb = xb_all[keep]
    c = np.polyfit(xb, mean[keep], 2)
    vx = -c[1] / (2 * c[0])
    return dict(bins=int(keep.sum()), range=f"{xb[0]} to {xb[-1]}", quad=round(c[0], 4), vertex=round(vx, 2),
                peak_bin_of_mean=float(xb[np.argmax(mean[keep])] if sign < 0 else xb[np.argmin(mean[keep])]),
                holds=bool(np.sign(c[0]) == sign and xb[0] < vx < xb[-1]))

xmv = np.arange(-5, 5, .25)
print("optimal:", shape_x(mv, 0, -1, xmv))
for c in ["movement (a)", "movement (b)"]:
    print(f"{c}: r2 {r2tab[c].mean():.4f} +/- {r2tab[c].sem():.3f}; vs pupil difference {np.mean(r2tab[c] - r2tab.pupil):+.3f}, "
          f"p = {stats.wilcoxon(r2tab[c], r2tab.pupil).pvalue:.3g}")
fig, ax = plt.subplots(figsize=(5, 3.5))
for k, c in zip(range(3), ["teal", "firebrick", "orange"]):
    ax.plot(xmv, np.nanmean(mv[:, k, :], axis=0), color=c)
ax.set_xlabel("movement index (bins of 0.25)")
ax.set_ylabel("mean P(state) over mice")
plt.tight_layout()
plt.show()
print("x from -5 to 4.75; y in [0, 1]; bins with fewer than 7 mice are included in the plot but not in the fit.")

**Result: Reproduced.** The mean P(optimal) over movement-index bins is an inverted U (negative quadratic term, vertex at 0.28, inside -4.25 to 3.75). Its r2 is 0.574 +/- 0.057 with the plotting bins (a), within 0.63 +/- 0.06, and 0.533 with 50 bins (b), outside; it does not differ from pupil's under either reading (p = 0.22 and 0.17). The (a) value is close to the lower bound of 0.57: over 10 shuffled splits per mouse it is 0.573 (b: 0.522).

> The probability of the disengaged state is a U-shaped function of the movement index.

In [ ]:
print("disengaged:", shape_x(mv, 1, +1, xmv))

**Result: Reproduced.** The mean P(disengaged) over movement-index bins has a positive quadratic term with its vertex (0.08) inside the range of bins with at least 7 mice (-4.25 to 4.0).

> The movement index is equally correlated with face motion energy and with locomotion speed, and highly correlated with both in each mouse.

In [ ]:
cr = []
for subj in subjects:
    g = fig3_set(subj)
    cr.append({"subject": subj, "r face": stats.pearsonr(g.movement, g.face_energy)[0],
               "r locomotion": stats.pearsonr(g.movement, g.running_speed)[0]})
cr = pd.DataFrame(cr).set_index("subject")
cr["difference"] = cr["r face"] - cr["r locomotion"]
print(cr.round(4).to_string())
print(f"\nlargest |difference| {cr.difference.abs().max():.4f}; smallest r {cr[['r face', 'r locomotion']].min().min():.3f}; "
      f"(a) both above 0.7 in every mouse: {bool((cr[['r face', 'r locomotion']] > .7).all().all())}; "
      f"(b) above 0.5: {bool((cr[['r face', 'r locomotion']] > .5).all().all())}")

**Result: Reproduced.** In every mouse the movement index correlates equally with face motion energy and with locomotion speed (difference below 0.0001), with r from 0.674 to 0.855, so reading (b), both above 0.5, holds; reading (a), above 0.7, fails in one mouse (BW071, 0.674).

## Raw measures by state (claims `fig3-distance-from-optimal-pupil`, `fig3-raw-face-energy-trend`, `fig3-raw-locomotion-no-difference`)

Per mouse, the mean at stimulus onset over the trials of each state, sub-optimal states pooled; sessions read both ways of O3: (a) sessions with more than 10 optimal-state trials and a post hoc pupil (Figure3.py), (b) all included sessions. Each measure's trials are those where it is present.

In [ ]:
def by_state(col, sess_col, fn=lambda g, c: g[c]):
    rows = []
    for subj in subjects:
        g = trials[(trials.subject == subj)]
        if sess_col:
            g = g[g[sess_col]]
        v = fn(g, col)
        st = g.hmm_state
        rows.append({"subject": subj, "optimal": v[st == 0].mean(), "disengaged": v[st == 1].mean(), "sub-optimal": v[st > 1].mean()})
    return pd.DataFrame(rows).set_index("subject")

def report(tab, scale, larger=None):
    m_, se_ = tab.mean() * scale, tab.sem() * scale
    print("   mean +/- SEM:", {k: f"{m_[k]:.3f} +/- {se_[k]:.3f}" for k in tab.columns})
    for other in ["disengaged", "sub-optimal"]:
        w = stats.wilcoxon(tab.optimal, tab[other])
        print(f"   optimal vs {other}: difference {np.mean(tab.optimal - tab[other]) * scale:+.3f}, "
              f"optimal lower in {int((tab.optimal < tab[other]).sum())} of 13, p = {w.pvalue:.2g}")

dist = lambda g, c: (g[c] - g.subject.map(optimal_pupil)).abs()
for label, sc in [("(a) sessions with >10 optimal trials and pupil", "sess_opt10"), ("(b) all included sessions", None)]:
    print("Absolute difference from the optimal pupil (% of session maximum),", label)
    report(by_state("post_hoc_pupil_diameter", sc, dist), 100)

**Result: Partially reproduced.** The distance from the optimal pupil is smallest in the optimal state and lower than in the disengaged (13 of 13 mice, p = 0.00024 under both session readings) and sub-optimal states (p = 0.00024 under (a), 0.0012 under (b)), but the values are larger than the paper's: 8.8%, 16.0% and 11.8% of the session maximum under (a) against 7%, 12.6% and 9%.

In [ ]:
for label, sc in [("(a)", "sess_opt10"), ("(b)", None)]:
    print("Face motion energy (a.u.)", label)
    report(by_state("face_energy", sc), 1)

**Result: Reproduced.** Under reading (a), face motion energy is 0.482 +/- 0.024 a.u. in the optimal state, 0.441 in the disengaged and 0.448 in the sub-optimal states, all within tolerance; the optimal mean is the largest, above sub-optimal with p = 0.017, and above disengaged with p = 0.19 (paper 0.06). Under reading (b) the values are 0.485, 0.435 and 0.453 (p = 0.048 and 0.08).

In [ ]:
for label, sc in [("(a)", "sess_opt10"), ("(b)", None)]:
    print("Locomotion speed (m/s)", label)
    report(by_state("running_speed", sc), 1)

**Result: Reproduced.** Locomotion speed is 0.136, 0.137 and 0.136 m/s in the optimal, disengaged and sub-optimal states (reading (a)), and neither test finds a difference (p = 0.84 and 0.89; reading (b) p = 0.68 and 0.95).

## Variability of the measures (claims `fig4a-lower-variability`, `fig4b-variability-at-transitions`)

> The 10-trial variability of pupil diameter, face motion energy and locomotion speed is lower in the optimal state than in the disengaged and sub-optimal states.

As Figure4.py: all included sessions, trials with the three 10-trial standard deviations and the pupil present; per mouse the mean of each standard deviation over the trials of each state, sub-optimal states pooled.

In [ ]:
STD3 = ["post_hoc_pupil_std10", "face_std_10", "running_std10"]
fig4 = {}
for subj in subjects:
    g = trials[trials.subject == subj].dropna(subset=STD3 + ["post_hoc_pupil_diameter"]).copy()
    g["movement"] = stats.zscore(g.face_energy) + stats.zscore(g.running_speed)
    fig4[subj] = g
n_tests = 0
for meas in STD3:
    tab = pd.DataFrame({subj: {"optimal": g.loc[g.hmm_state == 0, meas].mean(), "disengaged": g.loc[g.hmm_state == 1, meas].mean(),
                               "sub-optimal": g.loc[g.hmm_state > 1, meas].mean()} for subj, g in fig4.items()}).T
    print(meas, "per mouse:")
    print(tab.round(4).T.to_string())
    print(meas, "mean over mice:", tab.mean().round(4).to_dict())
    for other in ["disengaged", "sub-optimal"]:
        w = stats.wilcoxon(tab.optimal, tab[other])
        n_tests += (w.pvalue < 0.05) and (tab.optimal.mean() < tab[other].mean())
        print(f"   optimal vs {other}: difference {np.mean(tab.optimal - tab[other]):+.4f}, optimal lower in "
              f"{int((tab.optimal < tab[other]).sum())} of 13, p = {w.pvalue:.2g}")
print("tests with the optimal state lower at p < 0.05:", int(n_tests), "of 6")

**Result: Partially reproduced.** All six differences have the stated sign, and five of the six tests give p < 0.05; locomotion speed variability in the optimal state against the sub-optimal states does not (p = 0.057).

> Variability of the measures decreases on entering the optimal state and increases on leaving it.

As Figure4.py: optimal epochs from flex_hmm.get_dwell_times on all included sessions with a code dwell above 9; entries from a discrete non-optimal epoch and exits to one; the 10-trial standard deviation over the 10 trials before and the 10 after the boundary, averaged per mouse; the test compares position 10 (the last trial before the entry, or the last optimal trial) with position 20.

In [ ]:
enter_means, exit_means = {m: [] for m in STD3}, {m: [] for m in STD3}
n_trans = {}
for subj in subjects:
    g = trials[trials.subject == subj].sort_values(["path", "trial_in_session"])
    E = {m: [] for m in STD3}
    X_ = {m: [] for m in STD3}
    ne = nx = 0
    for path, sg in g.groupby("path"):
        dt = hulsey.dwell_table(sg.hmm_state.values)
        for SI in np.where(dt.state.values == 0)[0]:
            if dt.dwell.iloc[SI] <= 9:
                continue
            start, end = dt.first_trial.iloc[SI], dt.last_trial.iloc[SI]
            prior = dt.state.iloc[SI - 1] if SI > 0 else np.nan
            nxt = dt.state.iloc[SI + 1] if SI < len(dt) - 1 else np.nan
            for m in STD3:
                v = sg[m].values
                a, b = v[start - 10:start + 10], v[end - 10:end + 10]
                if prior > 0 and len(a) == 20:
                    E[m].append(a)
                if nxt > 0 and len(b) == 20:
                    X_[m].append(b)
            ne += prior > 0
            nx += nxt > 0
    for m in STD3:
        enter_means[m].append(np.nanmean(E[m], axis=0))
        exit_means[m].append(np.nanmean(X_[m], axis=0))
    n_trans[subj] = (ne, nx)
print("transitions per mouse (entries, exits):", n_trans)
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for i, m in enumerate(STD3):
    en, ex = np.array(enter_means[m]), np.array(exit_means[m])
    we, wx = stats.wilcoxon(en[:, 9], en[:, 19]), stats.wilcoxon(ex[:, 9], ex[:, 19])
    print(f"{m}: entry {np.mean(en[:, 9]):.4f} -> {np.mean(en[:, 19]):.4f} (lower after in {int((en[:, 19] < en[:, 9]).sum())} of 13, p = {we.pvalue:.2g}); "
          f"exit {np.mean(ex[:, 9]):.4f} -> {np.mean(ex[:, 19]):.4f} (higher after in {int((ex[:, 19] > ex[:, 9]).sum())} of 13, p = {wx.pvalue:.2g})")
    ax[i].plot(np.arange(-10, 10), (en - en.mean(1, keepdims=True)).mean(0), label="entry")
    ax[i].plot(np.arange(-10, 10), (ex - ex.mean(1, keepdims=True)).mean(0), label="exit")
    ax[i].axvline(0, color="k", lw=.5)
    ax[i].set_title(m, fontsize=9)
ax[0].legend()
ax[0].set_ylabel("mean-subtracted 10-trial SD")
ax[1].set_xlabel("trials from the boundary")
plt.tight_layout()
plt.show()

**Result: Partially reproduced.** All three measures' variability falls on entering the optimal state and rises on leaving it on average. Of the four comparisons the figure marks, pupil exit (p = 0.0012), locomotion entry (p = 0.04) and locomotion exit (p = 0.0012) give p < 0.05, and face entry does not (p = 0.11).

## Coupling of pupil and movement (claims `fig4d-pupil-movement-decoupling`, `figs6-individual-measure-decoupling`)

> The correlation between pupil diameter and the movement index is lower during the optimal state than during the disengaged and sub-optimal states.

As Figure4.py: the trial set of `fig4a`, per mouse and state the Pearson r between movement index and pupil diameter.

In [ ]:
rr = pd.DataFrame({subj: {"optimal": np.corrcoef(g.loc[g.hmm_state == 0, "movement"], g.loc[g.hmm_state == 0, "post_hoc_pupil_diameter"])[0, 1],
                          "disengaged": np.corrcoef(g.loc[g.hmm_state == 1, "movement"], g.loc[g.hmm_state == 1, "post_hoc_pupil_diameter"])[0, 1],
                          "sub-optimal": np.corrcoef(g.loc[g.hmm_state > 1, "movement"], g.loc[g.hmm_state > 1, "post_hoc_pupil_diameter"])[0, 1]}
                   for subj, g in fig4.items()}).T
print(rr.round(3).to_string())
print("mean:", rr.mean().round(3).to_dict())
for a_, b_ in [("optimal", "disengaged"), ("optimal", "sub-optimal"), ("disengaged", "sub-optimal")]:
    print(f"{a_} vs {b_}: difference {np.mean(rr[a_] - rr[b_]):+.3f}, first lower in {int((rr[a_] < rr[b_]).sum())} of 13, "
          f"p = {stats.wilcoxon(rr[a_], rr[b_]).pvalue:.2g}")

**Result: Reproduced.** The pupil-movement correlation is 0.077 in the optimal state against 0.439 in the disengaged and 0.267 in the sub-optimal states, lower in 13 and 12 of 13 mice (p = 0.00024 and 0.00049).

> The correlations of pupil diameter with face motion energy and with locomotion speed are also reduced during the optimal state.

As FigureS6.py: sessions of M5 with the six measures present; per mouse and state the Pearson r.

In [ ]:
for a_m, b_m in [("face_energy", "post_hoc_pupil_diameter"), ("running_speed", "post_hoc_pupil_diameter"), ("running_speed", "face_energy")]:
    tab = {}
    for subj in subjects:
        g = fig3_set(subj)
        tab[subj] = {nm: np.corrcoef(g.loc[sel, a_m], g.loc[sel, b_m])[0, 1]
                     for nm, sel in [("optimal", g.hmm_state == 0), ("disengaged", g.hmm_state == 1), ("sub-optimal", g.hmm_state > 1)]}
    tab = pd.DataFrame(tab).T
    print(f"r({a_m}, {b_m}) per mouse:")
    print(tab.round(3).T.to_string())
    print(f"r({a_m}, {b_m}) mean over mice:", tab.mean().round(3).to_dict())
    for other in ["disengaged", "sub-optimal"]:
        print(f"   optimal vs {other}: difference {np.mean(tab.optimal - tab[other]):+.3f}, optimal lower in "
              f"{int((tab.optimal < tab[other]).sum())} of 13, p = {stats.wilcoxon(tab.optimal, tab[other]).pvalue:.2g}")

**Result: Reproduced.** The correlation of pupil with face motion energy is lower in the optimal state than in the disengaged (p = 0.0024) and sub-optimal states (p = 0.00049); with locomotion speed it is lower than in the disengaged state (p = 0.0017) and not significantly lower than in the sub-optimal states (p = 0.068).

## Trial history and evoked pupil (claim `figs1-iti-independence`)

> The inter-trial interval is long enough that trial-evoked responses do not carry over; correlations of the measures at trial start with the prior ITI are weak and centered on zero, and evoked pupil responses do not differ across performance states.

As FigureS1.py on all included sessions: the ITI is the difference between consecutive trial start times within a session; per mouse, Pearson r of each measure at stimulus onset with the preceding ITI (the pupil value at onset is used for the code's trace sample 1 ms later); the evoked response is the signed larger of the largest rise and fall of pupil within 2.5 s of onset; per mouse the median per group, and a Kruskal-Wallis test over mice.

In [ ]:
itr, ev = [], []
for subj in subjects:
    g = trials[trials.subject == subj].sort_values(["path", "trial_in_session"]).copy()
    g["iti"] = g.groupby("path").start_time.diff()
    g = g.dropna(subset=["evoked_pupil", "post_hoc_pupil_diameter", "iti", "running_speed", "face_energy"])
    itr.append({"subject": subj, **{m: stats.pearsonr(g.iti, g[m])[0] for m in ["post_hoc_pupil_diameter", "running_speed", "face_energy"]}})
    ev.append({"subject": subj, "optimal": g.loc[g.hmm_state == 0, "evoked_pupil"].median() * 100,
               "disengaged": g.loc[g.hmm_state == 1, "evoked_pupil"].median() * 100,
               "sub-optimal": g.loc[g.hmm_state > 1, "evoked_pupil"].median() * 100,
               "indeterminate": g.loc[g.hmm_state.isna(), "evoked_pupil"].median() * 100})
itr = pd.DataFrame(itr).set_index("subject")
ev = pd.DataFrame(ev).set_index("subject")
print("r with the preceding ITI:")
print(itr.round(3).to_string())
for m in itr.columns:
    print(f"{m}: median r {itr[m].median():+.3f}, Wilcoxon against 0 p = {stats.wilcoxon(itr[m]).pvalue:.2g}; "
          f"(a) |median| < 0.1: {abs(itr[m].median()) < .1}, (b) < 0.2: {abs(itr[m].median()) < .2}")
print("\nmedian evoked pupil response per mouse (% of max):")
print(ev.round(2).to_string())
kw4 = stats.kruskal(*[ev[c] for c in ev.columns])
kw3 = stats.kruskal(*[ev[c] for c in ["optimal", "disengaged", "sub-optimal"]])
print(f"Kruskal-Wallis, four groups: H = {kw4.statistic:.2f}, p = {kw4.pvalue:.2g}; three discrete states (variant): p = {kw3.pvalue:.2g}")
print("group means:", ev.mean().round(2).to_dict())

**Result: Reproduced.** The median correlations of pupil, locomotion and face motion energy with the preceding ITI are -0.044, -0.007 and -0.044, none different from zero (p = 0.13, 0.89, 0.27), within reading (a). The median evoked pupil responses do not differ across the four groups (Kruskal-Wallis p = 0.22; three discrete states p = 0.15).

## Regression of state probabilities on the measures (claims `figs8a-regression-weights`, `figs8b-pupil-movement-interaction`)

> In linear regressions of state probability on the measures, the quadratic weights of pupil diameter and of locomotion speed are negative for the optimal state and positive for the disengaged state, as are the weights of their 10-trial SDs, while face motion energy weights center on zero.

As FigureS8.py: sessions of M5 with the six measures; per mouse, OLS of each state's posterior on the min-max scaled design (constant, pupil P, face F, locomotion L, their squares and products, their 10-trial SDs, and P*F*L; the design scaled again); second model on pupil, movement index M, their squares and product, the pupil SD and the movement SD index.

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
names1 = ["1", "P", "F", "L", "P*P", "P*F", "P*L", "F*F", "F*L", "L*L", "Pstd", "Fstd", "Lstd", "P*F*L"]
names2 = ["1", "P", "M", "P*P", "P*M", "M*M", "Pstd", "Mstd"]
W1, W2 = [], []
for subj in subjects:
    g = fig3_set(subj)
    mm = MinMaxScaler()
    X = mm.fit_transform(g[["post_hoc_pupil_diameter", "face_energy", "running_speed"]].values)
    S = mm.fit_transform(g[["post_hoc_pupil_std10", "face_std_10", "running_std10"]].values)
    X2 = mm.fit_transform(np.c_[g.post_hoc_pupil_diameter.values, g.movement.values])
    pf = PolynomialFeatures(degree=2)
    A1 = mm.fit_transform(np.c_[pf.fit_transform(X), S, X[:, 0] * X[:, 1] * X[:, 2]])
    A1[:, 0] = 1
    A2 = mm.fit_transform(np.c_[pf.fit_transform(X2), g.post_hoc_pupil_std10.values, g.movement_std.values])
    A2[:, 0] = 1
    for k in range(n_states[subj]):
        yk = g[f"state_{k + 1}_prob"].values
        grp = "optimal" if k == 0 else "disengaged" if k == 1 else "sub-optimal"
        W1.append({"subject": subj, "state": grp, **dict(zip(names1, sm.OLS(yk, A1).fit().params))})
        W2.append({"subject": subj, "state": grp, **dict(zip(names2, sm.OLS(yk, A2).fit().params))})
W1, W2 = pd.DataFrame(W1), pd.DataFrame(W2)
rows = []
for grp in ["optimal", "disengaged"]:
    w = W1[W1.state == grp]
    for term in ["P*P", "L*L", "F*F", "Pstd", "Lstd", "Fstd"]:
        rows.append({"state": grp, "term": term, "median": w[term].median(), "mean": w[term].mean(),
                     "positive": int((w[term] > 0).sum()), "p (vs 0)": stats.wilcoxon(w[term]).pvalue})
s8 = pd.DataFrame(rows)
print(s8.round(3).to_string(index=False))

**Result: Reproduced.** For the optimal state, the quadratic and SD weights of pupil and locomotion are negative (Wilcoxon against zero p at most 0.027) and positive for the disengaged state (p at most 0.033), while the face weights do not differ from zero (p from 0.191 to 0.839), under reading (a) and so under (b).

> With the movement index, the quadratic movement weight is negative for the optimal state, and the pupil-by-movement interaction is the only consistent weight for sub-optimal states (0.42 +/- 0.11), differing from its weight for the optimal state (-0.95 +/- 0.21).

In [ ]:
for grp in ["optimal", "disengaged", "sub-optimal"]:
    w = W2[W2.state == grp]
    print(grp, f"({len(w)} fits): mean +/- SEM", {t: f"{w[t].mean():.2f} +/- {w[t].sem():.2f}" for t in names2[1:]})
    print("   Wilcoxon vs 0 p:", {t: round(stats.wilcoxon(w[t]).pvalue, 3) for t in names2[1:]})
pm_s, pm_o = W2[W2.state == "sub-optimal"]["P*M"], W2[W2.state == "optimal"]["P*M"]
print(f"P*M sub-optimal {pm_s.mean():.2f} +/- {pm_s.sem():.2f} vs optimal {pm_o.mean():.2f} +/- {pm_o.sem():.2f}; "
      f"rank-sum p = {stats.ranksums(pm_s, pm_o).pvalue:.2g}")
print(f"M*M optimal mean {W2[W2.state == 'optimal']['M*M'].mean():.2f}")

**Result: Reproduced.** The pupil-by-movement weight is 0.41 +/- 0.10 over the 21 sub-optimal fits (expected 0.42 +/- 0.11) and -0.95 +/- 0.21 over the optimal fits (expected -0.95 +/- 0.21), different by a rank-sum test (p = 3.8e-06), and the mean quadratic movement weight of the optimal fits is negative (-1.30). It is the only sub-optimal weight that differs from zero (p < 0.001).

## Decoding the state from the measures (claims `fig5c-state-classification`, `figs7-pupil-best-raw-measure`, `fig5c-movement-index-best`)

> For every mouse, support vector machines on the value and 10-trial variability of the measures predict the performance state above chance, with 88.2% +/- 1.8% accuracy for optimal against disengaged and 77.7% +/- 2% for optimal against sub-optimal states.

As 5_decoding.py: sessions of M5; discrete-state trials with the six measures; the movement index and its SD z-scored over these trials; for each mouse, optimal against each other state when the two classes give more than 30 trials; at most 1500 trials per class, balanced, sampled at random; features min-max scaled; RBF SVC; 5-fold shuffled KFold; accuracy is the mean over folds and runs. In a run, the condition's columns are permuted across the training trials. The paper runs 1000 runs per condition; here the unshuffled condition needs one run (it does not vary), the shuffled conditions 3 runs, and the null (all columns shuffled) 20 runs, to fit the time limit (see the deviations). z = (accuracy - null mean) / null SD.

In [ ]:
N_RUN_COND, N_RUN_NULL = 3, 20
rng5 = np.random.default_rng(5)
t0 = time.time()
clf = []
tasks, tmeta = [], []
for subj in subjects:
    g = trials[(trials.subject == subj) & trials.sess_opt10].dropna(subset=["hmm_state"] + MEAS6).copy()
    g["movement"] = stats.zscore(g.face_energy) + stats.zscore(g.running_speed)
    g["movement_std"] = stats.zscore(g.face_std_10) + stats.zscore(g.running_std10)
    for st in sorted(g.hmm_state.unique())[1:]:
        o = g[g.hmm_state == 0].sample(frac=1, random_state=int(rng5.integers(2**31)))
        t_ = g[g.hmm_state == st].sample(frac=1, random_state=int(rng5.integers(2**31)))
        n = min(len(o), len(t_), 1500)
        o, t_ = o.iloc[:n], t_.iloc[:n]
        y = np.r_[np.zeros(n), np.ones(n)]
        if len(y) <= 30:
            continue
        X = MinMaxScaler().fit_transform(np.r_[o[hulsey.DECODE_MEASURES].values, t_[hulsey.DECODE_MEASURES].values])
        cid = len(clf)
        clf.append({"subject": subj, "state": int(st), "type": "disengaged" if st == 1 else "sub-optimal",
                    "template": hulsey.STATE_NAMES[types[subj][int(st)]], "trials per class": n})
        from sklearn.model_selection import KFold
        for f, (a, b) in enumerate(KFold(5, shuffle=True, random_state=int(rng5.integers(2**31))).split(y)):
            for cond, cols in hulsey.SHUFFLE.items():
                nr = 1 if cond == 13 else N_RUN_NULL if cond == 14 else N_RUN_COND
                for r in range(nr):
                    tasks.append((X[a], y[a], X[b], y[b], cols, int(rng5.integers(2**31))))
                    tmeta.append((cid, cond, r, f))
res = hulsey.run_parallel(hulsey.svm_task, tasks)
print(f"{len(clf)} classifications, {len(tasks)} SVM fits in {(time.time() - t0) / 60:.1f} min")
acc5 = pd.DataFrame(tmeta, columns=["cid", "cond", "run", "fold"]).assign(acc=res)
per_run = acc5.groupby(["cid", "cond", "run"]).acc.mean()           # mean over folds
null = per_run.xs(14, level="cond").groupby("cid").agg(["mean", "std", lambda x: np.percentile(x, 95)])
null.columns = ["null mean", "null sd", "null p95"]
cond_acc = per_run.groupby(["cid", "cond"]).mean().unstack()
clf = pd.DataFrame(clf).join(null)
for cond in cond_acc.columns:
    clf[f"acc {cond}"] = cond_acc[cond] * 100
    clf[f"z {cond}"] = (cond_acc[cond] - clf["null mean"]) / clf["null sd"]
show = clf[["subject", "state", "type", "template", "trials per class", "acc 13", "acc 0", "null mean", "null sd", "z 13", "z 0"]].copy()
show["null mean"] *= 100
show["null sd"] *= 100
print(show.round(2).to_string(index=False))
for cond, label in [(13, "eight features unshuffled (condition 13)"), (0, "'All measures' of Figure5.py (condition 0)")]:
    for tp in ["disengaged", "sub-optimal"]:
        c = clf[clf.type == tp][f"acc {cond}"]
        print(f"{label}, optimal vs {tp}: {c.mean():.1f} +/- {c.sem():.1f}% over {len(c)} classifications")
    z = clf[f"z {cond}"]
    above95 = (clf[f"acc {cond}"] / 100 > clf["null p95"])
    print(f"   significant: (a) z > 1.98 {int((z > 1.98).sum())} of {len(z)}, (b) z > 1.645 {int((z > 1.645).sum())}, "
          f"(c) above the null's 95th percentile {int(above95.sum())}")

**Result: Reproduced.** With the eight features unshuffled, accuracy is 88.9% +/- 1.7% for optimal against disengaged (13 classifications) and 78.4% +/- 1.8% against sub-optimal states (20), within tolerance; the 'All measures' condition gives 88.5% and 77.7%. All 33 classifications are significant under every reading. Here 20 sub-optimal classifications are possible against 19 in the paper.

> Pupil diameter (value and variability) is the best single raw measure for state classification, better than face motion energy and only slightly better than locomotion speed.

Readings (a), Figure S7A: keep one measure's value and SD, shuffle all others (conditions 2 pupil, 3 face, 4 locomotion); (b), Figure S7B: shuffle one measure's value and SD with the movement index (conditions 6, 7, 8), comparing the drop from condition 13. Quantity (i) accuracy, (ii) z.

In [ ]:
for reading, (cp, cf, cl), drop in [("(a) keep one", (2, 3, 4), False), ("(b) shuffle one", (6, 7, 8), True)]:
    for q in ["acc", "z"]:
        P, F, L = (clf[f"{q} {c}"] for c in (cp, cf, cl))
        if drop:
            base = clf[f"{q} 13"]
            P, F, L = base - P, base - F, base - L   # a larger drop means more information
        wf, wl = stats.wilcoxon(P, F), stats.wilcoxon(P, L)
        print(f"{reading}, {q}{' drop' if drop else ''}: pupil {P.mean():.2f}, face {F.mean():.2f}, locomotion {L.mean():.2f}; "
              f"pupil > face in {int((P > F).sum())} of {len(P)}, p = {wf.pvalue:.2g}; pupil > locomotion in {int((P > L).sum())}, p = {wl.pvalue:.2g}")

**Result: Reproduced under reading (a).** Keeping one measure, pupil gives 71.5% against 66.1% for face motion energy (pupil higher in 24 of 33, p = 0.0011) and 68.1% for locomotion speed (higher on average, p = 0.053); reading (b) agrees (face p = 0.00029, locomotion p = 0.029).

> The movement index is the best predictor of performance state, encoding more information about states than pupil diameter.

Main reading (Figure 5C legend): keep the movement index value and SD, shuffle all others ("M only"), against keeping only the pupil value and SD (condition 2). Variants, as the criterion asks: condition 5 of 5_decoding.py (movement index kept with the pupil value) and condition 1 (face and locomotion shuffled), the label "Movement index" of the final Figure5.py. Both keep pupil information in the movement-index classifier, so they compare pupil and movement together against pupil alone and cannot show that the movement index carries more information than pupil.

In [ ]:
for label, cm in [("main: M only", "M only"), ("variant: condition 5", 5), ("variant: condition 1", 1)]:
    for q in ["acc", "z"]:
        M, P = clf[f"{q} {cm}"], clf[f"{q} 2"]
        w = stats.wilcoxon(M, P)
        print(f"{label}, {q}: movement {M.mean():.2f} vs pupil {P.mean():.2f}; movement higher in {int((M > P).sum())} of {len(M)}, p = {w.pvalue:.2g}")

**Result: Not reproduced.** Under the claim's main reading, keeping only the movement index and its SD gives 68.6% against 71.5% for the pupil alone, higher in 14 of 33 classifications (p = 0.086; z p = 0.068): the movement index is not better than pupil. The code's conditions shown as variants give the movement index higher accuracy (condition 5: 75.7%, p = 1.3e-05; condition 1: 77.9%, p = 2.3e-10), but they keep pupil information as well and do not test the claim.

## Summary

| Claim | Outcome |
|---|---|
| cohort-inclusion | partially reproduced (13 mice, 8 and 5 by task; 376 sessions pass, not 381) |
| training-duration | blocked: data missing |
| distractor-response-rate | reproduced (5.2% overall; one mouse at 16%) |
| figs2-n-states | reproduced (3 or 4 states; no mouse with 5) |
| fig1g-example-mouse-prediction | reproduced (4 states; 70.2% and 78.1%) |
| fig2a-state-types | reproduced, resting on an assumption for one state of BW059 |
| fig2b-optimal-occupancy | reproduced (indeterminate 13.1%) |
| fig2c-dwell-times | partially reproduced (tests and indeterminate dwell hold; optimal and sub-optimal dwells longer than 37 and 15) |
| fig2d-prediction-accuracy | reproduced (76.2%, gain 11.0 points) |
| fig3c-inverted-u-pupil | reproduced |
| fig3d-optimal-pupil-modality | reproduced (p = 0.013) |
| fig3e-u-disengaged-pupil | reproduced |
| figs4-r2-individual-movement | partially reproduced (lower on average; values and tests do not all hold under one reading) |
| fig3f-movement-index-inverted-u | reproduced (r2 0.574, close to the lower bound 0.57) |
| fig3f-movement-index-u-disengaged | reproduced |
| figs5-movement-index-correlations | reproduced |
| fig3-distance-from-optimal-pupil | partially reproduced (order and tests hold; values larger) |
| fig3-raw-face-energy-trend | reproduced |
| fig3-raw-locomotion-no-difference | reproduced |
| fig4a-lower-variability | partially reproduced (5 of 6 tests) |
| fig4b-variability-at-transitions | partially reproduced (signs hold; 3 of 4 marked tests) |
| fig4d-pupil-movement-decoupling | reproduced |
| figs6-individual-measure-decoupling | reproduced |
| fig5c-state-classification | reproduced (88.9% and 78.4%) |
| figs7-pupil-best-raw-measure | reproduced |
| fig5c-movement-index-best | not reproduced (movement index 68.6% against pupil 71.5%) |
| figs8a-regression-weights | reproduced |
| figs8b-pupil-movement-interaction | reproduced |
| figs1-iti-independence | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The GLM-HMM of the paper's ssm fork is reimplemented (`hulsey.py`), fit by the same EM with the same priors, initialisation and stopping rule; its random initial values differ from ssm's. The decoders use 3 shuffle runs per shuffled condition and 20 null runs, not 1000, to keep the notebook within its time limit. The sessions are the 376 that pass the paper's inclusion criteria, where the paper reports 381. Where the paper's state-typing code would score a state with no discrete trials as equal to every template, that state is typed on all sessions (one state of BW059).

**What was found about the dataset.** The dandiset holds the 13 included mice and 391 sessions of stages 5 and 6, more than the 381 the paper analysed, with no way to tell which were left out; the excluded mice, the other stage 5 and 6 sessions and the training sessions before stage 5 are not in it. `running_speed` is labelled cm/s but holds m/s. 38 of the 391 sessions have a post hoc pupil series that is entirely missing. The rig names are spelled several ways ('Box 1', 'Box1', 'Boox1'). No recording is deposited twice, and the measures share the clock of the trial table (data check).
